# Exact single-well variant design

This notebook designs one exact protein variant per well while purchasing or preparing each shared physical DNA part only once. Individual-oligo prices are integer eurocents; terminal WT-equivalent material and Echo transfers are not included in the monetary objective.

In [ ]:
# Install only when running in Google Colab.
try:
    import google.colab
    !apt-get update -qq
    !apt-get install -qq rustc cargo
    !pip install -q --no-warn-conflicts git+https://github.com/Fleishman-Lab/GGAssembler@exact_construct
except ImportError:
    pass

import json
from pathlib import Path

import numpy as np
import pandas as pd

from dawdlib.degenerate_dna.codon_selector import CodonSelector
from dawdlib.dijkstra.colorful import ShortestPathFinder
from dawdlib.exact_construct import (
    build_exact_variant_dnas,
    exact_variant_tables,
    read_sequence_space_variants,
)
from dawdlib.golden_gate.gate_data import GGData, reverse_complement
from dawdlib.golden_gate.gate_restriction import create_path_validator
from dawdlib.golden_gate.graph_maker import (
    GraphMaker,
    make_exact_variant_graph,
    make_length_tier_cost,
)
from dawdlib.golden_gate.utils import RequirementsFactory, parse_dna

## Inputs and experimental settings

The defaults use the checked-in MthUPO example. In Colab, upload the three input files and change `INPUT_DIR` to their location.

In [ ]:
INPUT_DIR = Path("example/MthUPO")
WT_DNA_FILE = INPUT_DIR / "MthUPO_wt.txt"
RESFILE = INPUT_DIR / "MthUPO_Sequence_Space.resfile"
VARIANTS_FILE = INPUT_DIR / "MthUPO_150_variants.csv"
OUTPUT_DIR = Path("exact_construct_output")

ORGANISM_ID = "37762"
PREFIX = "GACATTGGTCTCA"
SUFFIX = "TGAGACCAACGACGCCGTACTCTTTGTCAAC"
MIN_CONST_FRAGMENT_LENGTH = 15
MIN_FIDELITY = 0.1
MIN_FRAGMENTS = 2
MAX_FRAGMENTS = 26
COLORFUL_RETRIES = 20
OVERHANG_LENGTH = 4
ALLOW_WT_DIRECT_SYNTHESIS = False
PRIMER_LENGTH = 20
PRIMER_PRICE_PER_NT_CENTS = 13

## Individual-oligo pricing

The pricing function receives the complete physical ordered length. It returns integer eurocents or `None` when that length is unavailable. Replace this callable to use another vendor or product.

In [ ]:
individual_oligo_cost = make_length_tier_cost([
    (13, 60, 13),
    (61, 80, 17),
])

assert individual_oligo_cost(60) == 780
assert individual_oligo_cost(61) == 1037
assert individual_oligo_cost(81) is None

## Optional custom overhang filter

Collaborators can exclude overhangs already used in a plasmid assembly. The custom function can only remove overhangs; GGAssembler's normal efficiency, GC, invariance, and fidelity checks still apply. This example also excludes reverse complements intentionally. Leave `PLASMID_OVERHANGS` empty to apply no additional filtering.

In [ ]:
PLASMID_OVERHANGS = set()  # Example: {"AATG", "GCTT"}
excluded_overhangs = PLASMID_OVERHANGS | {
    reverse_complement(overhang) for overhang in PLASMID_OVERHANGS
}

def overhang_is_available(overhang):
    return overhang not in excluded_overhangs

## Build exact DNA variants and graph

In [ ]:
wt_dna = parse_dna(str(WT_DNA_FILE)).upper()
variants = read_sequence_space_variants(
    str(VARIANTS_FILE), str(RESFILE), wt_dna
)
variant_dnas = build_exact_variant_dnas(
    wt_dna, variants, CodonSelector(ORGANISM_ID), enzymes=["BsaI"]
)
print(f"Loaded {len(variant_dnas)} exact variants.")

requirements = RequirementsFactory(
    min_oligo_length=0,
    max_oligo_length=10_000,
    min_const_oligo_length=MIN_CONST_FRAGMENT_LENGTH,
    min_efficiency=0.25,
    oligo_prefix=PREFIX,
    oligo_suffix=SUFFIX,
    const_cost=0,
    min_fidelity=MIN_FIDELITY,
    filter_gc_overhangs=False,
)
ggdata = GGData(
    temperature=requirements.gg_temp,
    hours=requirements.gg_hours,
    min_efficiency=requirements.min_efficiency,
    min_fidelity=requirements.min_fidelity,
)
graph, source, target = make_exact_variant_graph(
    GraphMaker(ggdata),
    wt_dna,
    variant_dnas,
    requirements,
    segment_length_cost=individual_oligo_cost,
    primer_length=PRIMER_LENGTH,
    primer_price_per_nt_cents=PRIMER_PRICE_PER_NT_CENTS,
    allow_wt_direct_synthesis=ALLOW_WT_DIRECT_SYNTHESIS,
    overhang_filter=overhang_is_available,
    gatelength=OVERHANG_LENGTH,
)
print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

## Select a path with the existing search

This task intentionally does not change or claim optimality for GGAssembler's existing path search. Candidate paths are filtered with the existing global Golden Gate compatibility validator.

In [ ]:
path_is_valid = create_path_validator(ggdata)
path_finder = ShortestPathFinder(graph, ggdata, source, target)
np.random.seed(1)
candidates = []
for max_fragments in range(MIN_FRAGMENTS, MAX_FRAGMENTS + 1):
    for _ in range(COLORFUL_RETRIES):
        path = path_finder.find_shortest_path(
            len_cutoff=max_fragments, no_colors=max_fragments + 1
        )
        fragment_count = len(path) - 1
        if path and MIN_FRAGMENTS <= fragment_count <= max_fragments and path_is_valid(path):
            cost_cents = sum(
                graph.edges[first, second]["weight"]
                for first, second in zip(path[:-1], path[1:])
            )
            candidates.append((cost_cents, path))

if not candidates:
    raise RuntimeError("No compatible exact-construct path was found")

cost_cents, chosen_path = min(candidates, key=lambda candidate: candidate[0])
overhangs = [gate.bps for gate in chosen_path if not gate.src_or_target]
fidelity = ggdata.reaction_fidelity(*overhangs)._asdict()
print(f"Selected {len(chosen_path) - 1} fragments for €{cost_cents / 100:.2f}")
print("Junctions:", [(gate.idx, gate.bps) for gate in chosen_path if not gate.src_or_target])
print("Fidelity:", fidelity)

## Physical parts, variant BOM, and summary

In [ ]:
result = exact_variant_tables(
    chosen_path, graph, variant_dnas, fidelity_metrics=fidelity
)
display(result.parts)
display(result.bom)
display(pd.Series(result.summary, name="value"))

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
result.parts.to_csv(OUTPUT_DIR / "physical_parts.csv", index=False)
result.bom.to_csv(OUTPUT_DIR / "variant_bom.csv", index=False)
pd.DataFrame(chosen_path).to_csv(OUTPUT_DIR / "selected_path.csv", index=False)
with (OUTPUT_DIR / "summary.json").open("w") as handle:
    json.dump(result.summary, handle, indent=2)
print(f"Wrote results to {OUTPUT_DIR.resolve()}")